In [1]:
import tensorflow as tf
import numpy as np

---

In [203]:
# load MNIST
(x_train,y_train),(x_test,y_test)=tf.keras.datasets.fashion_mnist.load_data()#tf.keras.datasets.mnist.load_data()
x_train=np.reshape(x_train,(len(x_train),28,28,1)).astype('float32')/255.0-0.5
x_test=np.reshape(x_test,(len(x_test),28,28,1)).astype('float32')/255.0-0.5
y_train=tf.keras.utils.to_categorical(y_train,10)
y_test=tf.keras.utils.to_categorical(y_test,10)

In [204]:
class AttentiveConv2D(tf.keras.layers.Layer):
    def __init__(self,filters,kernel_size,activation,**kwargs):
        super().__init__(**kwargs)
        self.filters=filters
        self.kernel_size=kernel_size
        self.activation=activation
        self.conv2d=tf.keras.layers.Conv2D(filters=self.filters,kernel_size=self.kernel_size,activation=self.activation,strides=(1,1),padding='valid')

    def build(self,input_shape):
        self.attention_weights=self.add_weight(name='attention',shape=(self.filters,),initializer=tf.keras.initializers.Constant(1.0),trainable=True)
        super().build(input_shape)

    def call(self,inputs,use_attention=True):
        outputs=self.conv2d(inputs)
        attention=tf.nn.relu(self.attention_weights)
        outputs=outputs*attention
        return [outputs,attention]

    def get_config(self):
        config=super().get_config()
        config.update({'filters':self.filters,'kernel_size':self.kernel_size,'activation':self.activation})
        return config

In [207]:
# create model
x=tf.keras.layers.Input((28,28,1))
# 1
t,o1=AttentiveConv2D(filters=64,kernel_size=3,activation='relu')(x)
t=tf.keras.layers.MaxPooling2D(pool_size=(2,2))(t)
# 2
t,o2=AttentiveConv2D(filters=48,kernel_size=3,activation='relu')(t)
t=tf.keras.layers.MaxPooling2D(pool_size=(2,2))(t)
# 3
t,o3=AttentiveConv2D(filters=32,kernel_size=3,activation='relu')(t)
t=tf.keras.layers.MaxPooling2D(pool_size=(2,2))(t)
# Classifier
t=tf.keras.layers.Flatten()(t)
t=tf.keras.layers.Dense(32,activation='relu')(t)
y=tf.keras.layers.Dense(10,activation='softmax')(t)
# Model
model=tf.keras.Model(inputs=[x],outputs=[y,o1,o2,o3],name='test_model')
model.summary()

Model: "test_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_44 (InputLayer)          │ (None, 28, 28, 1)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ attentive_conv2d_90                  │ [(None, 26, 26, 64), (64)]  │             704 │
│ (AttentiveConv2D)                    │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_73 (MaxPooling2D)      │ (None, 13, 13, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ attentive_conv2d_91                  │ [(None, 11, 11, 48), (48)]  │          27,744 │
│ (AttentiveConv2D)                    │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_74 (MaxPooling2D)      │ (None, 5, 5, 48)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ attentive_conv2d_92                  │ [(None, 3, 3, 32), (32)]    │          13,888 │
│ (AttentiveConv2D)                    │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_75 (MaxPooling2D)      │ (None, 1, 1, 32)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_23 (Flatten)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_46 (Dense)                     │ (None, 32)                  │           1,056 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_47 (Dense)                     │ (None, 10)                  │             330 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 43,722 (170.79 KB)

 Trainable params: 43,722 (170.79 KB)

 Non-trainable params: 0 (0.00 B)

In [208]:
def l1_loss(y_true,y_pred):
    return tf.reduce_mean(tf.abs(y_pred))

z1=tf.zeros((len(x_train),64),dtype=np.float32)
z2=tf.zeros((len(x_train),48),dtype=np.float32)
z3=tf.zeros((len(x_train),32),dtype=np.float32)
# 1
optimizer=tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer,loss=[tf.keras.losses.CategoricalCrossentropy(),l1_loss,l1_loss,l1_loss,],loss_weights=[1.0,0.1,0.1,0.1],
    metrics=[tf.keras.metrics.CategoricalAccuracy(),tf.keras.metrics.MeanAbsoluteError(),tf.keras.metrics.MeanAbsoluteError(),
            tf.keras.metrics.MeanAbsoluteError()])
model.fit(x_train,[y_train,z1,z2,z3],epochs=5)
# 2
optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=optimizer,loss=[tf.keras.losses.CategoricalCrossentropy(),l1_loss,l1_loss,l1_loss,],loss_weights=[1.0,0.1,0.1,0.1],
    metrics=[tf.keras.metrics.CategoricalAccuracy(),tf.keras.metrics.MeanAbsoluteError(),tf.keras.metrics.MeanAbsoluteError(),
            tf.keras.metrics.MeanAbsoluteError()])
model.fit(x_train,[y_train,z1,z2,z3],epochs=5)
# 3 tuning process
model.layers[1].attention_weights._trainable=False
model.layers[3].attention_weights._trainable=False
model.layers[5].attention_weights._trainable=False
model.summary()
optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=optimizer,loss=[tf.keras.losses.CategoricalCrossentropy(),l1_loss,l1_loss,l1_loss,],loss_weights=[1.0,0.0,0.0,0.0],
    metrics=[tf.keras.metrics.CategoricalAccuracy(),tf.keras.metrics.MeanAbsoluteError(),tf.keras.metrics.MeanAbsoluteError(),
            tf.keras.metrics.MeanAbsoluteError()])
model.fit(x_train,[y_train,z1,z2,z3],epochs=10)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 31s 16ms/step - attentive_conv2d_90_loss: 0.8516 - attentive_conv2d_90_mean_absolute_error: 0.8516 - attentive_conv2d_91_loss: 0.8944 - attentive_conv2d_91_mean_absolute_error: 0.8944 - attentive_conv2d_92_loss: 0.8001 - attentive_conv2d_92_mean_absolute_error: 0.8001 - dense_47_categorical_accuracy: 0.7702 - dense_47_loss: 0.6237 - loss: 0.8783
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 30s 16ms/step - attentive_conv2d_90_loss: 0.5729 - attentive_conv2d_90_mean_absolute_error: 0.5729 - attentive_conv2d_91_loss: 0.7137 - attentive_conv2d_91_mean_absolute_error: 0.7137 - attentive_conv2d_92_loss: 0.6862 - attentive_conv2d_92_mean_absolute_error: 0.6862 - dense_47_categorical_accuracy: 0.8360 - dense_47_loss: 0.4484 - loss: 0.6456
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - attentive_conv2d_90_loss: 0.4173 - attentive_conv2d_90_mean_absolute_error: 0.4173 - attentive_conv2d_91_loss: 0.5750 - attentive_conv2d_91_mean_absolute_error: 0.57

Model: "test_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_44 (InputLayer)          │ (None, 28, 28, 1)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ attentive_conv2d_90                  │ [(None, 26, 26, 64), (64)]  │             704 │
│ (AttentiveConv2D)                    │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_73 (MaxPooling2D)      │ (None, 13, 13, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ attentive_conv2d_91                  │ [(None, 11, 11, 48), (48)]  │          27,744 │
│ (AttentiveConv2D)                    │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_74 (MaxPooling2D)      │ (None, 5, 5, 48)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ attentive_conv2d_92                  │ [(None, 3, 3, 32), (32)]    │          13,888 │
│ (AttentiveConv2D)                    │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_75 (MaxPooling2D)      │ (None, 1, 1, 32)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_23 (Flatten)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_46 (Dense)                     │ (None, 32)                  │           1,056 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_47 (Dense)                     │ (None, 10)                  │             330 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 131,168 (512.38 KB)

 Trainable params: 43,578 (170.23 KB)

 Non-trainable params: 144 (576.00 B)

 Optimizer params: 87,446 (341.59 KB)

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 31s 16ms/step - attentive_conv2d_90_loss: 0.2713 - attentive_conv2d_90_mean_absolute_error: 0.2713 - attentive_conv2d_91_loss: 0.3834 - attentive_conv2d_91_mean_absolute_error: 0.3834 - attentive_conv2d_92_loss: 0.5216 - attentive_conv2d_92_mean_absolute_error: 0.5216 - dense_47_categorical_accuracy: 0.8933 - dense_47_loss: 0.2911 - loss: 0.2911
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 27s 14ms/step - attentive_conv2d_90_loss: 0.2713 - attentive_conv2d_90_mean_absolute_error: 0.2713 - attentive_conv2d_91_loss: 0.3834 - attentive_conv2d_91_mean_absolute_error: 0.3834 - attentive_conv2d_92_loss: 0.5216 - attentive_conv2d_92_mean_absolute_error: 0.5216 - dense_47_categorical_accuracy: 0.8942 - dense_47_loss: 0.2891 - loss: 0.2891
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 28s 15ms/step - attentive_conv2d_90_loss: 0.2713 - attentive_conv2d_90_mean_absolute_error: 0.2713 - attentive_conv2d_91_loss: 0.3834 - attentive_conv2d_91_mean_absolute_error: 0

---

In [201]:
z1=tf.zeros((len(x_test),48),dtype=np.float32)
z2=tf.zeros((len(x_test),32),dtype=np.float32)
z3=tf.zeros((len(x_test),32),dtype=np.float32)
h=model.evaluate(x_test,[y_test,z1,z2,z3])

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - attentive_conv2d_84_loss: 1.2961 - attentive_conv2d_84_mean_absolute_error: 1.2961 - attentive_conv2d_85_loss: 1.2029 - attentive_conv2d_85_mean_absolute_error: 1.2029 - attentive_conv2d_86_loss: 1.0790 - attentive_conv2d_86_mean_absolute_error: 1.0790 - dense_43_categorical_accuracy: 0.8844 - dense_43_loss: 0.3153 - loss: 0.3152


In [202]:
w1=tf.nn.relu(model.layers[1].get_weights()[0])
w2=tf.nn.relu(model.layers[3].get_weights()[0])
w3=tf.nn.relu(model.layers[5].get_weights()[0])
print('Layer 1:',np.count_nonzero(w1==0))
print('Layer 2:',np.count_nonzero(w2==0))
print('Layer 3:',np.count_nonzero(w3==0))

Layer 1: 0
Layer 2: 0
Layer 3: 0


---